# Loading the Dataset

In [1]:
from datasets import load_dataset

ds = load_dataset("zefang-liu/phishing-email-dataset", split="train")
ds = ds.class_encode_column("Email Type")

split1 = ds.train_test_split(test_size=350, stratify_by_column="Email Type", seed=20)
sample1 = split1["test"]

sample1[0]

c:\Users\adidn\Documents\Uni\Year 3\Final Year Project\Holmes\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'Unnamed: 0': 1869,
 'Email Text': 'term paper stuart , i cannot open your attachment . vince kaminski',
 'Email Type': 1}

Email Type: 0 == Phishing

Email Type: 1 == Safe

# Fetching emails to use in few-shot prompting

In [2]:
split2 = split1['train'].train_test_split(test_size=4, stratify_by_column='Email Type', seed=20)

leftover_emails = split2['train']

random_emails_4 = split2['test']

random_emails_4[3]

{'Unnamed: 0': 10315,
 'Email Text': 'done nero , ado 0 be , allas , apple , corel , plnnacle system from $ 20 each fourth which who feel foot now northern cold most . super cheaap softwares & shiiip to all countrieswe have every popular softwares u need ! you name it normal : $ 299 . oo ; you saave $ 249 . oo adobe acrobat v 6 . o professional pc - my price : $ 1 oo ; normal : $ 449 . 95 ; you saave $ 349 . 95 & more more more softwares to choose from free we do have full range softwares : adobe , alias maya , autodesk , borland , corel , crystal reports . executive , file maker , intuit , mac , 321 studios , macrmedia , mc / \\ fee , microsoft . nero , pinnacle systems , powerquest , quark , red hat , riverdeep , roxio , symantec , vmware softwares spoken & 315 more popular titles for youcheckk out 315 more popular softwares on our siteguaaranteed super low prlce = = = ciick here to check out = = = water . road people',
 'Email Type': 0}

# Iteration 1: Initial Evaluation

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "prompt": f"""
                                                    Analyse this email for phishing indicators.
                            Email: "{email}"
                            
                            Return a raw JSON object with this structure:
                            {{
                                "is_phishing": boolean,
                                "risk_level": "Low" | "Medium" | "High",
                                "explanation": "Concise reason why."
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 1 - Initial Stage"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:4b



  0%|          | 1/350 [00:08<52:02,  8.95s/it]

Unexpected Error on email 1 (Attempt 1/5): 'is_phishing'
Unexpected Error on email 1 (Attempt 2/5): 'is_phishing'
Unexpected Error on email 1 (Attempt 3/5): 'is_phishing'
Unexpected Error on email 1 (Attempt 4/5): 'is_phishing'


  1%|          | 2/350 [00:25<1:18:36, 13.55s/it]

Unexpected Error on email 2 (Attempt 1/5): 'is_phishing'
Unexpected Error on email 2 (Attempt 2/5): 'is_phishing'
Unexpected Error on email 2 (Attempt 3/5): 'is_phishing'
Unexpected Error on email 2 (Attempt 4/5): 'is_phishing'


...

Unexpected Error on email 250 (Attempt 1/5): 'is_phishing'


 75%|███████▌  | 264/350 [56:35<17:20, 12.10s/it]

Unexpected Error on email 264 (Attempt 1/5): 'charmap' codec can't encode character '\U00100ce3' in position 297: character maps to <undefined>


100%|██████████| 350/350 [1:14:32<00:00, 12.78s/it]


Analysis complete.
Results saved to 'Model Evaluation 1 - Initial Stage\gemma3_12b.csv'



# Iteration 2: Two-Shot Prompting

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "prompt": f"""
                                You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                                You must return a raw JSON object and absolutely nothing else.

                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Safe Email ---
                                Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                                Output:
                                {{
                                    "is_phishing": false,
                                    "risk_level": "Low",
                                    "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                                }}

                                --- EXAMPLE 2: Phishing Email ---
                                Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                                Output:
                                {{
                                    "is_phishing": true,
                                    "risk_level": "High",
                                    "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                Return a raw JSON object with this exact structure:
                                {{
                                    "is_phishing": boolean,
                                    "risk_level": "Low" | "Medium" | "High",
                                    "explanation": "Concise reason why."
                                }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [95]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 2 - Few-Shot Stage"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:4b



 17%|█▋        | 61/350 [05:06<22:13,  4.61s/it] 

Unexpected Error on email 61 (Attempt 1/5): 'is_phishing'


 33%|███▎      | 115/350 [09:43<19:20,  4.94s/it]

Unexpected Error on email 115 (Attempt 1/5): 'is_phishing'


...


Analysis complete.
Results saved to 'Model Evaluation 2 - Few-Shot Stage\qwen3.5_4b.csv'


Starting phishing email analysis...
Email count: 350
Model: gemma3:12b



 49%|████▊     | 170/350 [36:22<33:34, 11.19s/it]  

JSON Error on email 170 (Attempt 1/5). Hallucination: {
    "is_phishing": true,
    "risk_level": "High...


100%|██████████| 350/350 [1:15:03<00:00, 12.87s/it]


Analysis complete.
Results saved to 'Model Evaluation 2 - Few-Shot Stage\gemma3_12b.csv'



# Iteration 3: System Prompt Parameter, Increasing Precision and JSON Order

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. Do not flag automated newsletters or poor grammar as phishing unless there is clear malicious intent.
                            2. Look for credential harvesting, urgent financial requests, or suspicious URLs.
                            3. If ambiguous, default to Safe ("is_phishing": false).
                            4. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [13]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 3 - System Parameter and Increasing Precision"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:4b



  0%|          | 0/350 [00:00<?, ?it/s]

Unexpected Error on email 0 (Attempt 1/5): 'is_phishing'


  2%|▏         | 6/350 [01:01<47:17,  8.25s/it]  

Unexpected Error on email 6 (Attempt 1/5): 'is_phishing'


...

Unexpected Error on email 245 (Attempt 1/5): 'charmap' codec can't encode character '\u0144' in position 726: character maps to <undefined>


 88%|████████▊ | 308/350 [2:32:49<19:58, 28.53s/it]  

JSON Error on email 308 (Attempt 1/5). Hallucination: {
  "explanation": "The email is a forwarded quote...


100%|██████████| 350/350 [2:51:53<00:00, 29.47s/it]


Analysis complete.
Results saved to 'Model Evaluation 3 - System Parameter and Increasing Precision\gemma3_12b.csv'



# Iteration 4: Prompt Engineering to Increase Recall



In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [5]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 4 - Increasing Recall"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:4b



  7%|▋         | 23/350 [03:05<44:11,  8.11s/it] 

Unexpected Error on email 23 (Attempt 1/5): 'is_phishing'


 12%|█▏        | 42/350 [05:40<40:32,  7.90s/it]

Unexpected Error on email 42 (Attempt 1/5): 'is_phishing'


...

JSON Error on email 276 (Attempt 1/5). Hallucination: {
  "explanation": "The email exhibits several cha...


 93%|█████████▎| 327/350 [4:41:54<11:18, 29.52s/it]  

Unexpected Error on email 327 (Attempt 1/5): 'is_phishing'


100%|██████████| 350/350 [4:57:45<00:00, 51.05s/it] 


Analysis complete.
Results saved to 'Model Evaluation 4 - Increasing Recall\gemma3_12b.csv'



# Iteration 5: Lowering Temperature to 0.3

In [8]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.3
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [9]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 5 - Decreasing Temperature"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:4b



  0%|          | 0/350 [00:00<?, ?it/s]

Unexpected Error on email 0 (Attempt 1/5): 'is_phishing'


 15%|█▌        | 54/350 [06:56<35:49,  7.26s/it] 

Unexpected Error on email 54 (Attempt 1/5): 'is_phishing'


...

JSON Error on email 204 (Attempt 1/5). Hallucination: {
  "explanation": "The email exhibits multiple si...


100%|██████████| 350/350 [2:10:11<00:00, 22.32s/it]



Analysis complete.
Results saved to 'Model Evaluation 5 - Decreasing Temperature\qwen3.5_4b.csv'


Starting phishing email analysis...
Email count: 350
Model: gemma3:12b



 18%|█▊        | 63/350 [37:36<2:51:19, 35.82s/it]


KeyboardInterrupt: 

# Iteration 6: Five-Shot Prompting to Target Dead Spots

## Finding dead spots/ blind spots and identifying patterns

In [9]:
import os
import csv

def search_dead_spots(folder_list, model_list):
    # Dictionary to keep track of failures per email ID
    # Format: { 'email_id': {'count': 0, 'is_phishing': bool, 'risk_counts': {'High': 2, 'Low': 1}} }
    failure_tracker = {}
    
    for folder in folder_list:
        for model_name in model_list:
            safe_model_name = model_name.replace(':', '_')
            file_path = os.path.join(folder, f"{safe_model_name}.csv")
            
            if not os.path.exists(file_path):
                print(f"Skipping missing file: {file_path}")
                continue
                
            with open(file_path, mode='r', encoding='utf-8', errors='replace') as file:
                reader = csv.DictReader(file)
                
                for row in reader:
                    email_id = row['Email']
                    label = str(row['Label']).strip().lower()
                    verdict = str(row['is_phishing_verdict']).strip().lower()
                    
                    # Grab the risk level and normalize it (e.g., 'high', 'HIGH' -> 'High')
                    risk_level = str(row['risk_level']).strip().title()
                    
                    if verdict == 'error':
                        continue
                        
                    target_phishing_label = '0' 
                    
                    is_actual_phishing = (label == target_phishing_label)
                    predicted_phishing = (verdict == 'true')
                    
                    # Check if the model got it wrong
                    if is_actual_phishing != predicted_phishing:
                        if email_id not in failure_tracker:
                            failure_tracker[email_id] = {
                                'count': 0, 
                                'is_phishing': is_actual_phishing,
                                'risk_counts': {} # New dictionary to tally risk levels
                            }
                        
                        # Increment the overall failure count
                        failure_tracker[email_id]['count'] += 1
                        
                        # Increment the specific risk level count for this email
                        if risk_level not in failure_tracker[email_id]['risk_counts']:
                            failure_tracker[email_id]['risk_counts'][risk_level] = 0
                        failure_tracker[email_id]['risk_counts'][risk_level] += 1

    # Convert tracker dictionary to the requested list of dictionaries
    dead_spots = []
    for email_id, data in failure_tracker.items():
        # Find the most frequent risk level by checking which key has the highest value
        risk_counts = data['risk_counts']
        if risk_counts:
            majority_risk = max(risk_counts, key=risk_counts.get)
        else:
            majority_risk = "Unknown"

        dead_spots.append({
            'email': email_id,
            'misidentified_no': data['count'],
            'is_phishing': data['is_phishing'],
            'majority_risk_level': majority_risk
        })
        
    # Sort the list in descending order by the number of times it was misidentified
    dead_spots.sort(key=lambda x: x['misidentified_no'], reverse=True)
    
    # Slice the top 50
    top_50_dead_spots = dead_spots[:50]
    
    # Print the results nicely
    print(f"\n=== Top {len(top_50_dead_spots)} Dead Spots ===")
    for spot in top_50_dead_spots:
        print(spot)
        
    return top_50_dead_spots


folders = [
    "Model Evaluation 1 - Initial Stage", 
    "Model Evaluation 2 - Two-Shot Prompting", 
    "Model Evaluation 3 - System Parameter and Increasing Precision", 
    "Model Evaluation 4 - Increasing Recall", 
    "Model Evaluation 5 - Decreasing Temperature"
]
models = ["gemma3:4b", "qwen3.5:4b", "llama3_latest", "gemma3:12b"]

dead_spots = search_dead_spots(folders, models)
print(dead_spots)


=== Top 50 Dead Spots ===
{'email': '10623', 'misidentified_no': 17, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '1527', 'misidentified_no': 16, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '16451', 'misidentified_no': 16, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '16030', 'misidentified_no': 16, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '6518', 'misidentified_no': 15, 'is_phishing': True, 'majority_risk_level': 'Low'}
{'email': '12139', 'misidentified_no': 15, 'is_phishing': True, 'majority_risk_level': 'Low'}
{'email': '9019', 'misidentified_no': 14, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '1249', 'misidentified_no': 14, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '13612', 'misidentified_no': 14, 'is_phishing': True, 'majority_risk_level': 'Low'}
{'email': '10951', 'misidentified_no': 13, 'is_phishing': False, 'majority_risk_level': 'High'}
{'email': '1581', 'misi

In [23]:
def get_email_by_id(email_list, target_id):
    """Searches the dataset for a specific email ID."""
    
    for email in email_list:
        # We convert both to strings just in case the CSV read the ID as text 
        # but the original dataset stored it as an integer.
        if str(email['Unnamed: 0']) == str(target_id):
            return email
            
    return None  # Returns None if the email somehow isn't found

email = get_email_by_id(sample1, 6518)
print(email)  # Replace 12345 with the actual email ID you want to retrieve

{'Unnamed: 0': 6518, 'Email Text': 'returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output', 'Email Type': 0}


In [13]:
i = 1
for x in dead_spots:
    email = get_email_by_id(sample1, x['email'])
    print(f"Email {i}:")
    print(email)
    print(len(email['Email Text']))
    print("")
    i += 1

Email 1:
{'Unnamed: 0': 10623, 'Email Text': 'please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below .', 'Email Type': 1}
242

Email 2:
{'Unnamed: 0': 1527, 'Email Text': 'hpl nom for september 23 , 2000 ( see attached file : hplo 923 . xls ) - hplo 923 . xls', 'Email Type': 1}
87

Email 3:
{'Unnamed: 0': 16451, 'Email Text': 'Message-----Subject: please give generously> > > > > > > > Please give generously... > > > > URGENT - DUDLEY EARTHQUAKE APPEAL > > > > At 00:54 on Monday 23 September an earthquake measuring 4.8 on the > > > > Richter scale hit Dudley,UK causing untold disruption and distress - > > > > * Many were woken well before their giro arrived > > > > * Several priceless collections of mementos from the Balearics and > > > > Spanish costas were damaged > > > > * Three areas of historic and 

In [22]:
split2 = split1['train'].train_test_split(test_size=100, stratify_by_column='Email Type', seed=20)
sample2 = split2['test']

for email in sample2:
    if email['Email Text'] is None:
        continue
    if len(email['Email Text']) < 200:
        print(email)
        print(len(email['Email Text']))
        print("")

{'Unnamed: 0': 17101, 'Email Text': 'just for fun - atto 0136 . gif', 'Email Type': 1}
30

{'Unnamed: 0': 3717, 'Email Text': 'empty', 'Email Type': 0}
5

{'Unnamed: 0': 10519, 'Email Text': 'URL: http://www.newsisfree.com/click/-6,8572785,215/\nDate: 2002-10-05T03:05:34+01:00*Sport: *BBC Panorama programme, The Corruption of Racing, reveals little that \nis new, writes *Greg Wood*.\n', 'Email Type': 1}
192

{'Unnamed: 0': 3727, 'Email Text': 'entex contact list 1 / 12 / 00 per request of rita wynne x 3 - 7694 please see attachement and note that joanie ngo was added to the list . many thanks pamela chambers - thompson x 39680', 'Email Type': 1}
186

{'Unnamed: 0': 4977, 'Email Text': "Looks useful. Hopefully, they'll put up some more material soon.\thttp://ocw.mit.edu/global/all-courses.html", 'Email Type': 1}
107

{'Unnamed: 0': 11554, 'Email Text': 'URL: http://www.newsisfree.com/click/-4,8264811,1717/\nDate: 2002-09-26T05:19:10+01:00(whatever-dude)\n', 'Email Type': 1}
101

{'Unnam

## Prompt Execution

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "explanation": "Technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link. While the email has some spelling errors, it does not contain any of the common indicators of phishing such as suspicious links, requests for sensitive information, or unrealistic offers. The tone and content are consistent with a legitimate technical support request.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "explanation": "Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly big unrealistic financial incentive ('$ 15 , 000') for a 'pre-qualified degree'. Too good to be true. Directs the victim to a suspicious, unofficial domain. The email also mentions 'limited on time offer' which is a common tactic to create urgency and pressure the recipient into acting without due consideration. All of these are strong indicators of a phishing attempt.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 3: Phishing Email ---
                            Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                            Output:
                            {{
                                "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 4: Safe Email ---
                            Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                            Output:
                            {{
                                "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- EXAMPLE 6: Phishing Email ---
                            Email: "uc viagra at 0 . 95 $ per dose ! ! ! special offer ! jaru buy it !"
                            Output:
                            {{
                                "explanation": "This is a blatant pharmaceutical spam/phishing email. It employs classic spam indicators, including aggressive punctuation ('! ! !'), high-pressure sales language ('special offer !'), and unbelievably low pricing for restricted medication. Furthermore, the inclusion of random, nonsensical letter strings ('uc', 'jaru') is a known technique designed to confuse and bypass Bayesian spam filters by altering the text's mathematical probability.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        full_text = raw_response if raw_response else thinking_text

                        start_idx = full_text.find('{')
                        end_idx = full_text.rfind('}')
                        
                        # If both brackets exist and are in the correct order, slice the string
                        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                            json_string = full_text[start_idx:end_idx + 1]
                        else:
                            # Force an error to trigger the retry loop if no brackets are found
                            raise ValueError("No JSON brackets found in the AI response.")

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 5 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
# model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b']
model_list = ['gemma3:12b']
email_list = sample1
save_folder = "Model Evaluation 6 - Five-Shot Prompting to Target Dead Spots"

test_models(email_list, model_list, save_folder)

===Models selected: ['gemma3:12b']===

Starting phishing email analysis...
Email count: 350
Model: gemma3:12b



 33%|███▎      | 115/350 [1:09:29<2:29:36, 38.20s/it]

Unexpected Error on email 115 (Attempt 1/5): No JSON brackets found in the AI response.
Unexpected Error on email 115 (Attempt 2/5): No JSON brackets found in the AI response.
Unexpected Error on email 115 (Attempt 3/5): No JSON brackets found in the AI response.


# Evaluation Metrics Calculator

In [3]:
import csv
import os

def evaluate_models(model_list, save_folder):
    print(f"===Evaluating models: {model_list}===")
    os.makedirs(save_folder, exist_ok=True)

    model_comparison_file = os.path.join(save_folder, "model_comparison.csv")
    with open(model_comparison_file, mode='w', newline='') as comparison_file:
        writer = csv.writer(comparison_file)
        writer.writerow(["Model", "Accuracy", "Precision", "Recall", "F1 Score"])

        for model_name in model_list:

            fields = []
            prediction_categories = {
                'TP': 0,
                'FP': 0,
                'TN': 0,
                'FN': 0
            }
            metrics = {
                'Accuracy': None,
                'Precision': None,
                'Recall': None,
                'F1 Score': None
            }
                
            csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
            with open(csv_file, mode='r') as file:
                reader = csv.reader(file)

                fields = next(reader)
                total_rows = 0
                for row in reader:
                    if row[2] != "ERROR":
                        total_rows += 1

                    if row[1] == '0' and row[2] == 'True':
                        prediction_categories['TP'] += 1
                    elif row[1] == '0' and row[2] == 'False':
                        prediction_categories['FN'] += 1
                    elif row[1] == '1' and row[2] == 'True':
                        prediction_categories['FP'] += 1
                    elif row[1] == '1' and row[2] == 'False':
                        prediction_categories['TN'] += 1
            
            metrics['Accuracy'] = (prediction_categories['TP'] + prediction_categories['TN']) / total_rows
            metrics['Precision'] = prediction_categories['TP'] / (prediction_categories['TP'] + prediction_categories['FP'])
            metrics['Recall'] = prediction_categories['TP'] / (prediction_categories['TP'] + prediction_categories['FN'])
            metrics['F1 Score'] = 2 * (metrics['Precision'] * metrics['Recall']) / (metrics['Precision'] + metrics['Recall'])

            print(f"\nModel: {model_name}. Results evaluated: {total_rows}.\n")
            for key in metrics:
                print(f"\t{key}: {metrics[key]}")

            writer.writerow([model_name, metrics['Accuracy'], metrics['Precision'], metrics['Recall'], metrics['F1 Score']])

    print(f"\n===Model evaluation complete. Comparison saved to '{model_comparison_file}'===")

    
# prediction_categories
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
save_folder = "Model Evaluation 6 - Five-Shot Prompting to Target Dead Spots"
evaluate_models(model_list, save_folder)

===Evaluating models: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Model: gemma3:4b. Results evaluated: 327.

	Accuracy: 0.8623853211009175
	Precision: 0.7588235294117647
	Recall: 0.9699248120300752
	F1 Score: 0.8514851485148515

Model: llama3:latest. Results evaluated: 350.

	Accuracy: 0.9114285714285715
	Precision: 0.9083969465648855
	Recall: 0.8623188405797102
	F1 Score: 0.8847583643122676

Model: qwen3.5:4b. Results evaluated: 348.

	Accuracy: 0.882183908045977
	Precision: 0.7745664739884393
	Recall: 0.9852941176470589
	F1 Score: 0.8673139158576051

Model: gemma3:12b. Results evaluated: 106.

	Accuracy: 0.8679245283018868
	Precision: 0.7254901960784313
	Recall: 1.0
	F1 Score: 0.8409090909090908

===Model evaluation complete. Comparison saved to 'Model Evaluation 6 - Five-Shot Prompting to Target Dead Spots\model_comparison.csv'===
